# Part 0 (실습) — 환경 준비 & 첫 LangChain 호출

> **이 노트북의 목표**
> 이론에서 그린 지도 위에 첫 발을 디뎌봄. 아래 4단계를 거치며 "내 컴퓨터(또는 코랩)에서 LangChain이 Gemini를 부를 수 있는 상태"를 완성함.
>
> 1. 패키지 설치
> 2. 설치 확인
> 3. Gemini API 키 연결
> 4. 첫 호출 — "Hello, LangChain"
>
> **사용 모델**: Google Gemini (무료 등급으로 충분함)
> **실행 환경**: Google Colab 또는 로컬 Jupyter 둘 다 가능함

## 1단계. 패키지 설치

### 개념
LangChain은 **여러 개의 작은 패키지**로 쪼개져 있음. 필요한 부품만 골라 설치하는 구조임.

| 패키지 | 역할 |
|---|---|
| `langchain` | 핵심 본체. LCEL, `create_agent` 등이 들어 있음 (1.0+) |
| `langchain-google-genai` | Gemini 모델을 LangChain 블록으로 감싸주는 연결 패키지 |

### 동작 원리
`langchain-google-genai` 안에 들어 있는 `ChatGoogleGenerativeAI` 클래스가, Gemini API 호출을 LangChain의 공통 규격(`Runnable`)에 맞춰 포장해줌. 그래서 우리는 Gemini를 다른 LangChain 부품과 똑같은 방식(`.invoke()`)으로 다룰 수 있게 됨.

> 📌 **문법 메모**: `%pip install`은 노트북 안에서 패키지를 설치하는 매직 명령어임. 터미널에서는 `pip install ...`을 씀. `-q`는 출력을 줄이는 옵션(quiet)임.

In [ ]:
# LangChain 1.0 본체 + Gemini 연결 패키지 설치
%pip install -q -U langchain langchain-google-genai

## 2단계. 설치 확인

### 개념
설치가 끝나면, **버전을 찍어보는 것**이 첫 번째 디버깅 습관임. "분명 설치했는데 안 된다"의 절반은 버전이 다르거나, 설치가 다른 파이썬 환경에 들어간 경우임.

### 문법
- `import 패키지` → 패키지를 불러옴. 에러 없이 통과하면 설치 성공임.
- `패키지.__version__` → 설치된 버전 문자열을 돌려주는 약속된 속성(attribute)임.

In [ ]:
import langchain
import langchain_google_genai

print("langchain 버전:", langchain.__version__)
print("langchain-google-genai 설치 확인 완료")

## 3단계. Gemini API 키 연결

### 개념
LLM을 호출하려면 **API 키**가 필요함. API 키는 "이 요청은 내 계정에서 보낸 것"임을 증명하는 비밀번호 같은 것임.

발급: [Google AI Studio](https://aistudio.google.com/app/apikey) 에서 무료로 발급받음.

### 원리 — 왜 코드에 직접 붙여 넣지 않는가
키를 코드에 그대로 적으면(`api_key="AIza..."`) 깃허브에 올렸을 때 그대로 노출됨. 그래서 **환경 변수(environment variable)**에 넣고, 코드는 환경 변수를 읽기만 하게 함.

LangChain의 Gemini 블록은 `GOOGLE_API_KEY`라는 이름의 환경 변수를 **자동으로 찾아 읽음**. 그래서 변수 이름만 맞춰 두면 코드에 키를 쓸 필요가 없음.

> 📌 **참고**: 아래는 두 가지 방법을 모두 담음. 코랩이면 방법 A(getpass), 로컬이면 방법 B(.env)도 자주 씀. 둘 중 하나만 실행하면 됨.

In [ ]:
# 방법 A) 코랩/주피터 공통 — 실행하면 입력창이 떠서 키를 안전하게 받음
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키를 붙여넣으세요: ")

print("API 키 환경 변수 설정 완료:", "GOOGLE_API_KEY" in os.environ)

## 4단계. 첫 호출 — "Hello, LangChain"

### 개념
이제 진짜 첫 호출임. 흐름은 단 두 줄로 끝남.

1. **모델 블록을 만든다** → `ChatGoogleGenerativeAI(...)`
2. **질문을 넣어 실행한다** → `.invoke("...")`

### 핵심 문법: `ChatGoogleGenerativeAI`의 주요 인수

| 인수 | 의미 | 비유 |
|---|---|---|
| `model` | 어떤 Gemini 모델을 쓸지 (예: `"gemini-2.5-flash"`) | 어떤 직원을 부를지 |
| `temperature` | 0에 가까울수록 일관·보수적, 1에 가까울수록 창의·다양함 | 직원의 "자유분방함" 정도 |
| `max_retries` | 호출 실패 시 자동 재시도 횟수 | 전화 안 받으면 다시 거는 횟수 |

### 핵심 메서드: `.invoke()`
- **의미**: "입력 하나를 넣고, 결과 하나를 즉시 받는다"는 가장 기본 실행 메서드임.
- **동작**: 입력 문자열을 Gemini에 보내고, 응답을 `AIMessage` 객체로 돌려줌.
- `.content` 속성에 실제 답변 텍스트가 들어 있음.

> 📌 모든 LangChain 부품이 `.invoke()`를 공유함(= Runnable 규격). 이것이 나중에 파이프(`|`)로 줄줄이 잇는 LCEL의 토대가 됨. 지금 이 `.invoke()`가 강의 전체를 관통하는 첫 단추임.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 1) 모델 블록 만들기
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # 빠르고 저렴한 기본 모델
    temperature=0.7,            # 약간의 다양성 허용
    max_retries=2,              # 실패 시 2번까지 재시도
)

# 2) 첫 질문 던지기
response = llm.invoke("LangChain이 무엇인지 한국어로 두 문장으로 설명해줘.")

# 3) 응답 확인 — .content 에 답변 텍스트가 들어 있음
print(response.content)

### 응답 객체 들여다보기

`.invoke()`가 돌려준 것은 단순한 문자열이 아니라 **`AIMessage` 객체**임. 답변 본문 말고도 메타데이터(토큰 사용량 등)가 함께 들어 있음. 한 번 구조를 들여다보면 이후 디버깅이 쉬워짐.

In [ ]:
# 응답 객체의 타입과 주요 속성 살펴보기
print("타입:", type(response).__name__)
print("-" * 40)
print("답변 본문 (.content):")
print(response.content)
print("-" * 40)
print("토큰 사용량 (.usage_metadata):")
print(response.usage_metadata)

## ✏️ 미니 실습

아래 빈칸을 채워 직접 호출해봄. (정답 코드는 바로 아래 셀에 있음 — 먼저 스스로 해보고 비교할 것)

**과제**: `temperature`를 `0`으로 바꾼 모델을 새로 만들고, "리리마켓이라는 라이브커머스 브랜드의 한 줄 슬로건을 만들어줘" 라고 물어보기. `temperature=0`일 때 답이 어떻게 달라지는지 관찰해봄.

In [ ]:
# TODO: 여기에 직접 작성해보세요
# llm_strict = ChatGoogleGenerativeAI(model="...", temperature=...)
# answer = llm_strict.invoke("...")
# print(answer.content)

<details>
<summary>👉 정답 코드 보기</summary>

```python
llm_strict = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
answer = llm_strict.invoke("리리마켓이라는 라이브커머스 브랜드의 한 줄 슬로건을 만들어줘")
print(answer.content)
```
`temperature=0`은 가장 확률이 높은 답을 고르므로, 여러 번 돌려도 결과가 거의 일정함. 반대로 `temperature`를 올리면 매번 다른 슬로건이 나옴.
</details>

## 정리 — 무엇을 했는가

| 단계 | 한 일 | 핵심 |
|---|---|---|
| 1 | 패키지 설치 | `langchain`, `langchain-google-genai` |
| 2 | 설치 확인 | `import` + `__version__` |
| 3 | 키 연결 | `GOOGLE_API_KEY` 환경 변수 |
| 4 | 첫 호출 | `ChatGoogleGenerativeAI` → `.invoke()` → `.content` |

### 3줄 요약
1. LangChain은 부품 단위 패키지로 설치하며, Gemini는 `langchain-google-genai`로 붙임.
2. 모델 블록은 `ChatGoogleGenerativeAI(...)`로 만들고 `.invoke()`로 실행함.
3. 이 `.invoke()`가 모든 LangChain 부품이 공유하는 공통 규격이며, 다음 파트(LCEL)의 토대임.

### ▶ 다음 (Part 1)
Part 1에서는 메시지의 종류(System / Human / AI)를 구분하고, 프롬프트를 더 정교하게 설계하는 법으로 넘어감.